In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
#from imblearn.under_sampling import RandomUnderSampler
import pandas as pd
import xgboost
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [ ]:
data_path = 'C:\GaTech\MS-CSE\ISYE 6740\WiFOP\data\\final datasets\ca_5km_daily_panel_2020_2023_6_9 1\ca_5km_daily_panel_2020_2023_6_9.csv'
data = pd.read_csv(data_path, sep=',')
data.head()

In [ ]:
def median_imputation(df):
    for col in df.columns:
        if col == 'grid_id' or col == 'date':
            continue
        df[col] = df[col].fillna(df[col].median())
    return df

In [ ]:
#temporal train and test split

data['date'] = pd.to_datetime(data['date'], errors='raise')
data['year'] = data['date'].dt.year
train_mask = data['year'].isin([2020, 2021, 2022])
test_mask = data['year'] == 2023

In [ ]:
train_df = data.loc[train_mask].copy()
test_df = data.loc[test_mask].copy()

In [ ]:
#drop the grid id, date, w_lat, w_lon, n_lon, n_lat, year
X_train = train_df.drop(columns=['date', 'grid_id', 'w_lat', 'w_lon', 'n_lat', 'n_lon', 'year']).reset_index(drop=True)

X_test = test_df.drop(columns=['date', 'grid_id', 'w_lat', 'w_lon', 'n_lat', 'n_lon', 'year']).reset_index(drop=True)

In [ ]:
y_train = X_train.pop('fire_today')
y_test = X_test.pop('fire_today')

In [ ]:
#X_train, X_test, y_train, y_test = train_test_split(X_, y, stratify=y, test_size=0.20, random_state=42)

In [ ]:
X_train, X_test = median_imputation(X_train), median_imputation(X_test)

In [ ]:
#perform feature selection with xgboost
feature_selector = xgboost.XGBClassifier(device='cuda', n_estimators=400, random_state=42)
feature_selector.fit(X_train, y_train)

In [ ]:
importances = feature_selector.feature_importances_

In [ ]:
top_k = list(pd.DataFrame({'Features': X_train.columns, 'Importances': importances}).sort_values(by='Importances', ascending=False).iloc[:20, 0])

In [ ]:
top_k

In [ ]:
X_train_selected = X_train[top_k]
X_test_selected = X_test[top_k]

In [ ]:
X_train_selected.shape

In [ ]:
scaler2 = StandardScaler()
X_train_selected_scaled = scaler2.fit_transform(X_train_selected)
X_test_selected_scaled = scaler2.transform(X_test_selected)

In [ ]:
class Model(tf.keras.Model):
    def __init__(self, input_dimension, neurons, num_classes):
        super().__init__()

        #Convolutional Layer
        self.conv1 = tf.keras.layers.Conv1D(filters=32, kernel_size=3, activation='relu', padding='same')
        self.batch1 = tf.keras.layers.BatchNormalization()
        self.maxpool1 = tf.keras.layers.MaxPool1D(3)

        #Convolutional Layer
        self.conv2 = tf.keras.layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')
        self.batch2 = tf.keras.layers.BatchNormalization()
        self.maxpool2 = tf.keras.layers.MaxPool1D(3)

        #Flatten Layer
        self.flatten = tf.keras.layers.Flatten()

        #Dense Layer (hidden layer)
        self.dense = tf.keras.layers.Dense(units=neurons, activation='relu')
        self.batch3 = tf.keras.layers.BatchNormalization()
        self.output_layer = tf.keras.layers.Dense(units=num_classes, activation='sigmoid')

    def call(self, inputs):
        x = self.conv1(inputs)
        x = self.batch1(x)
        x = self.maxpool1(x)

        x = self.flatten(x)
        x = self.dense(x)
        x = self.batch3(x)
        output = self.output_layer(x)

        return output


In [ ]:
n_features = len(top_k)
model = Model(input_dimension=n_features, neurons=64, num_classes=1)
model.build(input_shape=(None, n_features, 1))
model.compile(loss=tf.keras.losses.BinaryCrossentropy(), optimizer='Adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
X_train_cnn = X_train_selected_scaled.reshape(-1, n_features, 1)
X_test_cnn = X_test_selected_scaled.reshape(-1, n_features, 1)

In [ ]:
num_classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=num_classes, y=y_train)
cw_dict = dict(zip(num_classes, weights))

In [ ]:
cw_dict

In [ ]:
cb = [EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]

history = model.fit(X_train_cnn, y_train, batch_size=100, epochs=20, verbose=2, validation_split=0.1, class_weight=cw_dict)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.history['loss']) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs, history.history['loss'], 'b', label='Training Loss')
plt.plot(epochs, history.history['val_loss'], 'r', label='Validation Loss')
plt.title('Training and Validation Loss - Feature Selected from XGBoost for CNN model - Jul 2023')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(epochs, history.history['accuracy'], 'b', label='Training accuracy')
plt.plot(epochs, history.history['val_accuracy'], 'r', label='Validation accuracy')
plt.title('Training and Validation Accuracy - Feature Selected from XGBoost for CNN model - Jul 2023')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
predictions = model.predict(X_test_cnn)
y_pred = np.argmax(predictions, axis=1)

In [ ]:
print(precision_score(y_test, y_pred, average='weighted', zero_division=0))
print(recall_score(y_test, y_pred, average='weighted', zero_division=0))
print(f1_score(y_test, y_pred, average='weighted', zero_division=0))

In [ ]:
clr = classification_report(y_test, y_pred, zero_division=0.0, output_dict=True)

In [ ]:
import seaborn as sns

df_clr = pd.DataFrame(clr).transpose()
plt.figure(figsize=(8,8))
sns.heatmap(df_clr[['precision', 'recall', 'f1-score']], annot=True)
plt.title('Classification Report - 4 Year Data')
plt.show()